In [37]:
import torch
import torch.nn.functional as f
import matplotlib.pyplot as plt
%matplotlib inline

In [48]:
block_size = 3
words = open('names.txt','r').read().splitlines()
chars = sorted(list(set("".join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.']=0
itos = {i:s for s,i in stoi.items()}
vocab_size = len(itos)
vocab_size

27

In [49]:
g = torch.Generator().manual_seed(2)

class Linear:
    def __init__(self,fan_in,fan_out):
        self.weight = torch.randn((fan_in,fan_out), generator = g)/fan_in**0.5
        self.bias = torch.zeros(fan_out) if bias else None

    def __call__(self,x):
        self.out = x@self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out
    
    def parameters(self):
        return [self.weight]+ ([] if self.bias in None else [self.bias])


In [50]:
class BatchNorm1D:
    def __init__(self,dim,esp=1e-5,momentum=0.1):
        self.eps = esp
        self.momentum = momentum
        self.training = True
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)
        self.running_var = torch.ones(dim)
        self.running_mean = torch.zeros(dim)

    def __call__(self,x):
        if self.training:
            xmean = x.mean(0,keepdim=True)
            xvar = x.var(0,keepdim=True)

        else:
            xmean = self.running_mean
            xvar = self.running_var
        xhat = x.var(0,keepdim=True, unbiased=True)
        self.out = (x-xmean)/torch.sqrt(xvar+self.eps)

        if self.training:
            self.running_var = (1-self.momentum)*self.running_var+self.momentum*xvar
            self.running_mean = (1-self.momentum)*self.running_mean+self.momentum*xmean

        return self.out

    def parameters(self):
        return [self.gamma,self.beta]


In [51]:
class Tanh:
    def __call__(self,x):
        self.out = torch.tanh(x)
        return self.out
    
    def parameters(self):
        return []

In [53]:
max_steps = 200_000
batch_size=32
lossi=[]
for i in range(max_steps):
    ix=torch.randint(0,Xtr.shape[0], (batch_size,), generator=g)
    Xb,Yb = Xtr[ix],ytr[ix]

    emb = C[Xb]
    x = emb.view(emb.shape[0],-1)
    for layer in layers:
        x=layer(x)

    loss = f.cross_entropy(x,Yb)

    for layer in layers:
        layer.out.retain_grad()
    for p in parameters:
        p.grad=None
    loss.backward()

    for p in parameters:
        p.data+=-lr*p.grad


    if i%10_000==0:
        print(f"{i}=> {loss.item():.4f}")
    lossi.append(loss.log10().item())


NameError: name 'Xtr' is not defined